# Fog-density detection — standalone Google Colab

All five project modules are embedded below; no repository checkout or code ZIP is required.
Select **Runtime → Change runtime type → GPU**, then run the cells in order.
The original full study remains 3 detectors × 2 arms × 3 seeds, with 30 epochs.
This can exceed a Colab session. Checkpoints go to Google Drive by default.

**Data:** Put the complete `fog-colab-data.zip` (including Foggy Cityscapes) in `MyDrive/fog-study/`.
Low-disk mode keeps the ZIP and outputs in Drive, extracts only one dataset copy locally,
and requires 15 GiB of free working space after extraction. It blocks additional dataset downloads.
Create it on your PC with `.venv/Scripts/python.exe scripts/package_colab_data.py`.
Pretrained model weights download when needed; extra dataset downloads are blocked in low-disk mode.
nuScenes cannot be anonymously downloaded by this notebook: its official download requires an account.
The data ZIP carries your existing prepared labels and referenced camera images, excluding lidar/sweeps/archive copies.
Use `--include-fog` when packaging to carry Foggy Cityscapes too.

See the [Colab FAQ](https://research.google.com/colaboratory/faq.html) and
[nuScenes downloads](https://www.nuscenes.org/download).

## Configuration — edit only this cell
Paths below are relative to MyDrive. In `zip` mode, the data folder contains the ZIP. In `directory` mode, it contains the extracted `data/` folder (and `.cache/` if present).

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil

# Storage: do not include /content/drive/MyDrive in these relative paths.
DRIVE_DATA_DIR = "fog-study"          # Example: "Research/Fog study"
DRIVE_OUTPUT_DIR = "fog-study/runs"   # Checkpoints, metrics and exported ZIPs
DATA_MODE = "zip"                    # "zip": extract locally; "directory": read extracted data in Drive
DATA_ZIP_NAME = "fog-colab-data.zip"
DATA_ZIP_URL = ""                    # Optional HTTPS URL; requires LOW_DISK_MODE=False
USE_DRIVE = True
LOW_DISK_MODE = True
MIN_FREE_GIB = 15                    # Runtime working-space reserve
RUNTIME_ROOT = "/content/fog-study"  # Code, caches, and ZIP-mode extracted data
RUN_NAME = "colab-full-study"         # Same name/settings to resume; new name for a changed experiment

# Study controls: defaults preserve the complete matched study.
ACTION = "train"                     # "train", "preflight", or "analyze"
STUDY = {
    "epochs": 30, "density_epochs": 15,
    "seeds": [42, 43, 44],
    "arms": ["rgb_control", "predicted_density"],
    "size": 640, "accumulate": 4, "save_every": 200, "repeats": 1,
    "learning_rates": {"YOLO11n": 1e-4, "YOLO11m": 1e-4, "DETR": 1e-5},
    "adapter_lr": 1e-3, "amp": True,
    "device": "cuda:0",
    "density_checkpoint": None,
}
# Relative model paths are resolved under RUNTIME_ROOT; DETR is a model ID.
MODEL_CHECKPOINTS = {
    "YOLO11n": "yolo11n.pt", "YOLO11m": "yolo11m.pt",
    "DETR": "facebook/detr-resnet-50",
}


## Initialize paths and storage — no settings to edit below

In [ ]:

def mydrive_path(relative):
    base = Path('/content/drive/MyDrive')
    from pathlib import PurePosixPath
    value = PurePosixPath(relative)
    if value.is_absolute() or '..' in value.parts or "\\" in str(relative) or ':' in str(relative):
        raise ValueError('Use a folder relative to MyDrive, such as Research/Fog study.')
    return base / value

if ACTION not in {'train', 'preflight', 'analyze'}:
    raise ValueError('ACTION must be train, preflight, or analyze.')
if DATA_MODE not in {'zip', 'directory'}:
    raise ValueError('DATA_MODE must be zip or directory.')
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {'.', '..'} or "\\" in RUN_NAME:
    raise ValueError('RUN_NAME must be a single folder name.')
if MIN_FREE_GIB < 0:
    raise ValueError('MIN_FREE_GIB cannot be negative.')
DRIVE_DATA_ROOT = mydrive_path(DRIVE_DATA_DIR)
DRIVE_RUNS_ROOT = mydrive_path(DRIVE_OUTPUT_DIR)
if Path(DATA_ZIP_NAME).name != DATA_ZIP_NAME or not DATA_ZIP_NAME or "\\" in DATA_ZIP_NAME:
    raise ValueError('DATA_ZIP_NAME must be a filename, not a path.')
if DATA_MODE == 'directory' and (not USE_DRIVE or DATA_ZIP_URL):
    raise ValueError('Directory mode requires mounted Drive and no DATA_ZIP_URL.')
ROOT = Path(RUNTIME_ROOT)
if not ROOT.is_absolute():
    raise ValueError('RUNTIME_ROOT must be an absolute runtime path.')
ROOT.mkdir(parents=True, exist_ok=True)
if LOW_DISK_MODE and (not USE_DRIVE or DATA_ZIP_URL):
    raise ValueError('Low-disk mode requires USE_DRIVE=True and DATA_ZIP_URL="".')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
DATA_ROOT = DRIVE_DATA_ROOT if DATA_MODE == 'directory' else ROOT
DATA_ZIP_PATH = DRIVE_DATA_ROOT / DATA_ZIP_NAME if USE_DRIVE else ROOT / DATA_ZIP_NAME
OUTPUT_ROOT = DRIVE_RUNS_ROOT if USE_DRIVE else ROOT / 'reports/fog'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print('Data root:', DATA_ROOT)
print('Dataset ZIP:', DATA_ZIP_PATH if DATA_MODE == 'zip' else '(not used)')
print('Checkpoints:', OUTPUT_ROOT / RUN_NAME)
print(f'Free runtime disk: {shutil.disk_usage(ROOT).free / 2**30:.1f} GiB')


## Restore embedded project code

In [ ]:
EMBEDDED_FILES = {'notebooks/_support/fog_analytics.py': '"""Saved-run analytics: tables, paired changes, per-class AP, density error and plots."""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\ndef export_analytics(output, rows=None):\n    output = Path(output)\n    if rows is None:\n        rows = json.loads((output/\'input-manifest.json\').read_text(encoding=\'utf-8\'))\n    frames, statuses = [], []\n    for directory in sorted(output.iterdir()):\n        if not directory.is_dir() or not (directory/\'status.json\').exists():\n            continue\n        status = json.loads((directory/\'status.json\').read_text())\n        statuses.append(status)\n        if status[\'status\'] == \'complete\' and (directory/\'results.csv\').exists():\n            frames.append(pd.read_csv(directory/\'results.csv\', dtype={\'density\': str}))\n    pd.DataFrame(statuses).to_csv(output/\'status.csv\', index=False)\n    if not frames:\n        raise RuntimeError(f\'No completed experiments under {output}. Check status.csv; resume training first.\')\n    results = pd.concat(frames, ignore_index=True)\n    results.to_csv(output/\'results.csv\', index=False)\n    keys = [\'model\', \'dataset\', \'seed\', \'density\']\n    control = results[results.arm == \'rgb_control\'][keys+[\'mAP50_95\',\'mean_ms\',\'p95_ms\']]\n    paired = results[results.arm != \'rgb_control\'].merge(control, on=keys, how=\'inner\',\n        suffixes=(\'\', \'_control\'), validate=\'many_to_one\')\n    paired[\'delta_mAP\'] = paired.mAP50_95-paired.mAP50_95_control\n    paired[\'delta_latency_ms\'] = paired.mean_ms-paired.mean_ms_control\n    paired.to_csv(output/\'paired-deltas.csv\', index=False)\n    metric_names = [\'mAP50_95\',\'AP50\',\'AR100\',\'precision_at_025\',\'recall_at_025\',\'mean_ms\',\'p95_ms\',\'serial_fps\']\n    seed_summary = results.groupby([\'dataset\',\'model\',\'arm\',\'density\'])[metric_names].agg([\'mean\',\'std\',\'count\'])\n    seed_summary.to_csv(output/\'seed-summary.csv\')\n    by_id = {r[\'id\']: r for r in rows}\n    density_rows = []\n    for path in output.glob(\'density-*/test-predictions.csv\'):\n        estimates = pd.read_csv(path)\n        for dataset, group in estimates.groupby(\'dataset\'):\n            error = group.predicted_beta-group.target_beta\n            density_rows.append({\'experiment\':path.parent.name,\'dataset\':dataset,\'images\':len(group),\n                                 \'MAE_beta\':float(error.abs().mean()),\'RMSE_beta\':float(np.sqrt((error**2).mean()))})\n    for path in output.glob(\'*-predicted_density-*/*-predictions.jsonl\'):\n        known = []\n        with path.open(encoding=\'utf-8\') as handle:\n            for line in handle:\n                if not line.endswith(\'\\n\'):\n                    break\n                row = json.loads(line)\n                target = by_id[row[\'id\']][\'beta\']\n                if target is not None:\n                    known.append((row[\'beta_input\'], target))\n        if known:\n            error = np.array([p-t for p,t in known])\n            density_rows.append({\'experiment\': path.parent.name, \'dataset\': path.stem.replace(\'-predictions\',\'\'),\n                \'images\': len(known), \'MAE_beta\': float(np.abs(error).mean()), \'RMSE_beta\': float(np.sqrt((error**2).mean()))})\n    pd.DataFrame(density_rows, columns=[\'experiment\',\'dataset\',\'images\',\'MAE_beta\',\'RMSE_beta\']).to_csv(\n        output/\'density-quality.csv\', index=False)\n    make_figures(output, results, paired)\n    return results\n\n\ndef make_figures(output, results, paired):\n    import matplotlib\n    matplotlib.use(\'Agg\', force=True)  # saved figures must work without Tcl/Tk or a desktop\n    import matplotlib.pyplot as plt\n    from fog_research import CLASSES\n    figures = output/\'figures\'\n    figures.mkdir(exist_ok=True)\n    plt.rcParams.update({\'font.size\': 10, \'figure.dpi\': 120})\n    styles = {\'rgb_control\': (\'#33658A\',\'o\'), \'predicted_density\': (\'#D17A22\',\'^\'), \'oracle_density\': (\'#548235\',\'s\')}\n    for dataset, frame in results.groupby(\'dataset\'):\n        overall = frame[frame.density == \'all\']\n        if overall.empty:\n            continue\n        fig, axes = plt.subplots(1,2,figsize=(14,5))\n        labels = []\n        for position, ((model,arm), part) in enumerate(overall.groupby([\'model\',\'arm\'])):\n            color, marker = styles[arm]\n            label = f\'{model}\\n{arm}\'\n            labels.append(label)\n            values = part.mAP50_95.to_numpy()\n            axes[0].bar(position, np.nanmean(values), color=color, alpha=.65)\n            axes[0].scatter(np.full(len(values),position),values,color=\'black\',marker=marker,s=20)\n            axes[1].scatter(part.mean_ms, part.mAP50_95, color=color, marker=marker, label=f\'{model}: {arm}\')\n        axes[0].set_xticks(range(len(labels)),labels,rotation=25,ha=\'right\')\n        axes[0].set(ylabel=\'mAP@[.50:.95]\',ylim=(0,1), title=\'Bars: seed mean; points: individual seeds\')\n        axes[1].set(xlabel=\'End-to-end mean latency (ms/image)\',ylabel=\'mAP@[.50:.95]\',ylim=(0,1),title=\'Accuracy versus latency\')\n        axes[1].legend(fontsize=7)\n        fig.suptitle(f\'{dataset} | held-out labeled images | completed runs only\')\n        fig.tight_layout(); fig.savefig(figures/f\'{dataset}-comparison.png\',bbox_inches=\'tight\'); plt.close(fig)\n        numeric = frame[~frame.density.isin([\'all\',\'unknown\'])].copy()\n        if not numeric.empty:\n            numeric[\'beta\'] = numeric.density.astype(float)\n            fig, ax = plt.subplots(figsize=(9,5))\n            for (model,arm), part in numeric.groupby([\'model\',\'arm\']):\n                mean = part.groupby(\'beta\').mAP50_95.mean()\n                ax.plot(mean.index,mean.values,marker=styles[arm][1],label=f\'{model}: {arm}\')\n            ax.set(xlabel=\'Known beta (m^-1)\',ylabel=\'Mean mAP@[.50:.95]\',ylim=(0,1),title=f\'{dataset}: accuracy by fog level\')\n            ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(figures/f\'{dataset}-density.png\'); plt.close(fig)\n        table = overall.groupby([\'model\',\'arm\'])[[f\'AP_{c}\' for c in CLASSES]].mean()\n        fig, ax = plt.subplots(figsize=(10,max(3,.6*len(table))))\n        plotted = ax.imshow(table.to_numpy(),vmin=0,vmax=1,aspect=\'auto\',cmap=\'viridis\')\n        ax.set_xticks(range(len(CLASSES)),CLASSES,rotation=30,ha=\'right\')\n        ax.set_yticks(range(len(table)),[f\'{m}: {a}\' for m,a in table.index])\n        ax.set_title(f\'{dataset}: per-class AP (blank = no ground truth)\')\n        fig.colorbar(plotted,ax=ax,label=\'AP@[.50:.95]\')\n        fig.tight_layout(); fig.savefig(figures/f\'{dataset}-classes.png\',bbox_inches=\'tight\'); plt.close(fig)\n        delta = paired[(paired.dataset == dataset) & (paired.density == \'all\')]\n        if not delta.empty:\n            fig, axes = plt.subplots(1,2,figsize=(12,4))\n            names = []\n            for index, ((model,arm), part) in enumerate(delta.groupby([\'model\',\'arm\'])):\n                names.append(f\'{model}\\n{arm}\')\n                for ax,column in zip(axes,[\'delta_mAP\',\'delta_latency_ms\']):\n                    ax.scatter(np.full(len(part),index),part[column],marker=styles[arm][1])\n            for ax in axes:\n                ax.axhline(0,color=\'black\',linewidth=.8)\n                ax.set_xticks(range(len(names)),names,rotation=20,ha=\'right\')\n            axes[0].set_ylabel(\'Paired delta mAP (positive = better)\')\n            axes[1].set_ylabel(\'Paired delta latency ms (positive = slower)\')\n            fig.suptitle(f\'{dataset}: matched seeds and test images; no significance claim\')\n            fig.tight_layout(); fig.savefig(figures/f\'{dataset}-deltas.png\',bbox_inches=\'tight\'); plt.close(fig)\n    for history_file in output.glob(\'*/history.csv\'):\n        history = pd.read_csv(history_file)\n        if not {\'train_loss\',\'val_loss\'} <= set(history.columns):\n            continue\n        fig, ax = plt.subplots(figsize=(7,4))\n        ax.plot(history.epoch,history.train_loss,label=\'Train\',marker=\'o\')\n        ax.plot(history.epoch,history.val_loss,label=\'Tuning\',marker=\'^\')\n        ax.set(xlabel=\'Epoch\',ylabel=\'Detector loss\',title=history_file.parent.name)\n        ax.legend(); fig.tight_layout(); fig.savefig(figures/f\'{history_file.parent.name}-loss.png\'); plt.close(fig)\n', 'notebooks/_support/fog_data.py': '"""Dataset entry points; nuScenes preparation lives in fog_prepare."""\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nfrom fog_research import CLASSES\n\n\ndef resolve_foggy_images(project_root, explicit=None, download=False):\n    """Reuse Windows/Linux Kaggle caches from the user\'s downloader; optional same-source download."""\n    import os\n    project_root = Path(project_root).resolve()\n    candidates = [Path(explicit)] if explicit else []\n    candidates += [project_root/\'data/fog/cityscapes\',\n                   Path.home()/\'Downloads\'/\'foggy-cityscapes\',\n                   Path.home()/\'Downloads\'/\'Foggy Cityscapes\',\n                   project_root/\'.cache/kagglehub/datasets/minhducnguyen9705/foggy-cityscapes/versions/1\',\n                   Path.home()/\'.cache/kagglehub/datasets/minhducnguyen9705/foggy-cityscapes/versions/1\']\n    for path in candidates:\n        if path.is_dir() and ((path/\'leftImg8bit_foggy\').is_dir() or (path/\'train\').is_dir()\n                              or next(path.rglob(\'leftImg8bit_foggy\'), None) is not None):\n            return path\n    if not download:\n        return None\n    # Keep downloads inside this workspace. Version 1 matches Untitled2.ipynb.\n    os.environ[\'KAGGLEHUB_CACHE\'] = str(project_root/\'.cache/kagglehub\')\n    try:\n        import kagglehub\n    except ImportError as error:\n        raise ImportError(\'Install requirements-fog-research.txt to enable the optional Kaggle download\') from error\n    print(\'Downloading the same Foggy Cityscapes v1 source as Untitled2.ipynb (~29.1 GB archive plus extraction).\', flush=True)\n    return Path(kagglehub.dataset_download(\'minhducnguyen9705/foggy-cityscapes/versions/1\'))\n\n\ndef cityscapes_manifest(root, output, with_detection_labels=False):\n    """Kaggle/official images suffice for density supervision; gtFine is optional.\n\n    Rows without gtFine are explicitly density-only, never negative detection images.\n    Set with_detection_labels=True for the optional Foggy Cityscapes detection benchmark.\n    """\n    root, output = Path(root).resolve(), Path(output)\n    if output.exists():\n        raise FileExistsError(f\'{output} exists. Reuse it or choose a new output path.\')\n    city_ids = {24:\'person\', 25:\'person\', 26:\'car\', 27:\'truck\', 28:\'bus\', 31:\'train\', 32:\'motorcycle\', 33:\'bicycle\'}\n    rows = []\n    from tqdm.auto import tqdm\n    # Kaggle extracts may nest leftImg8bit_foggy below additional directories.\n    image_roots = ([root/\'leftImg8bit_foggy\'] if (root/\'leftImg8bit_foggy\').is_dir() else\n                   ([root] if (root/\'train\').is_dir() else list(root.rglob(\'leftImg8bit_foggy\'))))\n    if len(image_roots) != 1:\n        raise ValueError(f\'Expected one Foggy Cityscapes image root under {root}; found {len(image_roots)}\')\n    image_root = image_roots[0]\n    for split in (\'train\',\'val\'):\n        paths = sorted((image_root/split).glob(\'*/*_leftImg8bit_foggy_beta_*.png\'))\n        cache = {}\n        for image in tqdm(paths, desc=f\'Foggy Cityscapes {split}\', mininterval=2):\n            base, suffix = image.name.split(\'_leftImg8bit_foggy_beta_\')\n            beta = float(suffix.removesuffix(\'.png\'))\n            mask_path = root/\'gtFine\'/split/image.parent.name/f\'{base}_gtFine_instanceIds.png\'\n            if base not in cache:\n                if with_detection_labels and not mask_path.is_file():\n                    raise FileNotFoundError(f\'Missing instance labels: {mask_path}\')\n                boxes, labels, crowds = [], [], []\n                with Image.open(image) as source:\n                    width, height = source.size\n                if with_detection_labels:\n                    with Image.open(mask_path) as source:\n                        mask = np.array(source)\n                    for instance in np.unique(mask):\n                        category = int(instance//1000 if instance >= 1000 else instance)\n                        if category in city_ids:\n                            yy, xx = np.where(mask == instance)\n                            boxes.append([int(xx.min()),int(yy.min()),int(xx.max())+1,int(yy.max())+1])\n                            labels.append(CLASSES.index(city_ids[category])); crowds.append(int(instance < 1000))\n                cache[base] = boxes, labels, crowds, width, height\n            boxes, labels, crowds, width, height = cache[base]\n            group = \'_\'.join(base.split(\'_\')[:2])\n            bucket = int(hashlib.sha256(group.encode()).hexdigest()[:8],16) % 10\n            if with_detection_labels:\n                assigned = \'test\' if split == \'val\' else (\'density_train\' if bucket < 2 else \'val\' if bucket < 4 else \'detector_train\')\n            else:\n                assigned = \'test\' if split == \'val\' else (\'val\' if bucket < 2 else \'density_train\')\n            rows.append({\'id\':f\'city/{base}/{beta}\', \'image\':str(image),\'dataset\':\'foggy_cityscapes\',\n                         \'group\':group,\'split\':assigned,\'beta\':beta,\'boxes\':boxes,\'labels\':labels,\'iscrowd\':crowds,\n                         \'width\':width,\'height\':height,\'annotation_source\':str(mask_path) if with_detection_labels else None,\n                         \'detection_labeled\':with_detection_labels,\n                         \'label_policy\':\'instance-mask bounding rectangles; rider merged into person\' if with_detection_labels else \'beta-only; not a detection target\'})\n    if not rows:\n        raise FileNotFoundError(f\'No official Foggy Cityscapes images under {root}/leftImg8bit_foggy\')\n    seen = set()\n    for row in list(rows):\n        base = row[\'id\'].split(\'/\')[1]\n        split = \'val\' if row[\'split\'] == \'test\' else \'train\'\n        clear = root/\'leftImg8bit\'/split/base.split(\'_\')[0]/f\'{base}_leftImg8bit.png\'\n        if clear.is_file() and base not in seen:\n            rows.append({**row,\'id\':f\'city/{base}/clear\',\'image\':str(clear),\'beta\':0.0})\n            seen.add(base)\n    from fog_prepare import atomic_json\n    atomic_json(output, rows)\n    return output\n\n\ndef nuscenes_manifest(root, output, version=\'v1.0-trainval\'):\n    from fog_prepare import prepare_nuscenes\n    return prepare_nuscenes(root, output, version=version)\n', 'notebooks/_support/fog_prepare.py': '"""Resumable nuScenes preparation without loading millions of sweep records into RAM."""\nimport hashlib\nimport json\nimport os\nimport sqlite3\nfrom collections import Counter\nfrom pathlib import Path\n\nimport ijson\nimport numpy as np\nfrom filelock import FileLock\nfrom pyquaternion import Quaternion\nfrom shapely.geometry import MultiPoint, box as rectangle\nfrom tqdm.auto import tqdm\n\nCAMERAS = (\'CAM_FRONT\', \'CAM_FRONT_RIGHT\', \'CAM_BACK_RIGHT\', \'CAM_BACK\', \'CAM_BACK_LEFT\', \'CAM_FRONT_LEFT\')\nSCHEMA = 3\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n    os.replace(temporary, path)\n\n\ndef stream(path):\n    with Path(path).open(\'rb\') as handle:\n        yield from ijson.items(handle, \'item\', use_float=True)\n\n\ndef source_signature(root, version):\n    names = (\'scene\', \'sample\', \'sample_data\', \'sample_annotation\', \'ego_pose\',\n             \'calibrated_sensor\', \'sensor\', \'instance\', \'category\')\n    return {\'schema\': SCHEMA, \'root\': str(root), \'version\': version,\n            \'tables\': {name: [((root/version)/f\'{name}.json\').stat().st_size,\n                              ((root/version)/f\'{name}.json\').stat().st_mtime_ns] for name in names}}\n\n\ndef metadata_index(root, version, cache):\n    signature = source_signature(root, version)\n    db_path = cache / \'metadata.sqlite\'\n    stamp = cache / \'metadata-source.json\'\n    if db_path.is_file() and stamp.is_file() and json.loads(stamp.read_text()) == signature:\n        return db_path\n    temp = cache / \'metadata-building.sqlite\'\n    # Recreate only this explicitly named disposable database after an interrupted import.\n    if temp.exists():\n        temp.unlink()\n    connection = sqlite3.connect(temp)\n    try:\n        connection.execute(\'PRAGMA journal_mode=OFF\')\n        connection.execute(\'PRAGMA synchronous=OFF\')\n        connection.execute(\'CREATE TABLE records(kind TEXT, token TEXT, parent TEXT, payload TEXT, PRIMARY KEY(kind,token))\')\n        camera_poses = set()\n        for kind in signature[\'tables\']:\n            batch = []\n            for row in tqdm(stream(root/version/f\'{kind}.json\'), desc=f\'Index {kind}\', unit=\' rows\', mininterval=2):\n                if kind == \'sample_data\':\n                    if not row[\'is_key_frame\'] or not row[\'filename\'].startswith(\'samples/CAM_\'):\n                        continue\n                    camera_poses.add(row[\'ego_pose_token\'])\n                if kind == \'ego_pose\' and row[\'token\'] not in camera_poses:\n                    continue\n                parent = row.get(\'sample_token\', row.get(\'scene_token\', \'\'))\n                batch.append((kind, row[\'token\'], parent, json.dumps(row)))\n                if len(batch) >= 5000:\n                    connection.executemany(\'INSERT INTO records VALUES(?,?,?,?)\', batch)\n                    batch.clear()\n            if batch:\n                connection.executemany(\'INSERT INTO records VALUES(?,?,?,?)\', batch)\n            connection.commit()\n        connection.execute(\'CREATE INDEX by_parent ON records(kind,parent)\')\n        connection.commit()\n    finally:\n        connection.close()\n    os.replace(temp, db_path)\n    atomic_json(stamp, signature)\n    return db_path\n\n\ndef clipped_bounds(points, width, height):\n    """Clip the projected convex hull, not its bounding rectangle (not equivalent)."""\n    if len(points) < 3 or not np.isfinite(points).all():\n        return None\n    lo, hi = points.min(axis=0), points.max(axis=0)\n    if hi[0] <= 0 or hi[1] <= 0 or lo[0] >= width or lo[1] >= height:\n        return None\n    hull = MultiPoint(points).convex_hull.intersection(rectangle(0, 0, width, height))\n    return list(hull.bounds) if not hull.is_empty and hull.area > 0 else None\n\n\ndef prepare_nuscenes(root, output, version=\'v1.0-trainval\'):\n    """Build complete scene shards with progress; resume after interruption.\n\n    Official train: deterministic 90% train / 10% tuning by scene. Official val:\n    held-out test. Every labeled keyframe and all six cameras are accounted for.\n    Missing files are recorded and block a full-data claim unless explicitly allowed.\n    """\n    from nuscenes.utils.splits import create_splits_scenes\n    from nuscenes.utils.data_classes import Box\n    from fog_research import CLASSES\n    root, output = Path(root).resolve(), Path(output).resolve()\n    cache = output.parent / (output.stem + \'-preparation\')\n    cache.mkdir(parents=True, exist_ok=True)\n    with FileLock(str(cache/\'prepare.lock\'), timeout=0):\n        db = metadata_index(root, version, cache)\n        connection = sqlite3.connect(db)\n        connection.execute(\'PRAGMA cache_size=-64000\')\n        def table(kind):\n            return {row[\'token\']: row for (payload,) in connection.execute(\n                \'SELECT payload FROM records WHERE kind=?\', (kind,)) for row in [json.loads(payload)]}\n        def children(kind, parent):\n            return [json.loads(p) for (p,) in connection.execute(\n                \'SELECT payload FROM records WHERE kind=? AND parent=?\', (kind, parent))]\n        def get(kind, token):\n            return json.loads(connection.execute(\'SELECT payload FROM records WHERE kind=? AND token=?\',\n                                                  (kind, token)).fetchone()[0])\n        sensors, calibrated = table(\'sensor\'), table(\'calibrated_sensor\')\n        categories, instances = table(\'category\'), table(\'instance\')\n        scenes = table(\'scene\')\n        splits = create_splits_scenes()\n        train_names = set(splits[\'mini_train\' if version == \'v1.0-mini\' else \'train\'])\n        test_names = set(splits[\'mini_val\' if version == \'v1.0-mini\' else \'val\'])\n        ordered = sorted([s for s in scenes.values() if s[\'name\'] in train_names | test_names], key=lambda r: r[\'name\'])\n        report = {\'source\': source_signature(root, version), \'expected_scenes\': len(train_names | test_names),\n                  \'selected_scenes\': len(ordered),\n                  \'expected_images\': 0, \'available_images\': 0, \'missing_images\': [], \'missing_channels\': []}\n        shard_paths = []\n        for scene in tqdm(ordered, desc=\'Project scenes (resumable)\', mininterval=2):\n            shard = cache / f\'{scene["name"]}.json\'\n            if shard.exists():\n                data = json.loads(shard.read_text(encoding=\'utf-8\'))\n                # Recompute scenes that previously had missing images: newly downloaded\n                # files become eligible on the next preparation run.\n                reusable = data.get(\'source\') == report[\'source\'] and not data[\'missing_images\']\n                reusable = reusable and all(Path(r[\'image\']).is_file() for r in data[\'rows\'])\n            else:\n                reusable = False\n            if not reusable:\n                bucket = int(hashlib.sha256(scene[\'name\'].encode()).hexdigest()[:8], 16) % 10\n                split = \'test\' if scene[\'name\'] in test_names else (\'val\' if bucket == 0 else \'detector_train\')\n                data = {\'rows\': [], \'expected_images\': 0, \'missing_images\': [], \'missing_channels\': [], \'source\': report[\'source\']}\n                for sample in sorted(children(\'sample\', scene[\'token\']), key=lambda r: r[\'timestamp\']):\n                    world_boxes = []\n                    for ann in children(\'sample_annotation\', sample[\'token\']):\n                        category = categories[instances[ann[\'instance_token\']][\'category_token\']][\'name\']\n                        cls = (\'person\' if category.startswith(\'human.pedestrian.\') and\n                               category.rsplit(\'.\', 1)[-1] not in (\'stroller\', \'wheelchair\', \'personal_mobility\') else\n                               \'bus\' if category.startswith(\'vehicle.bus.\') else\n                               {\'vehicle.car\': \'car\', \'vehicle.truck\': \'truck\', \'vehicle.bicycle\': \'bicycle\',\n                                \'vehicle.motorcycle\': \'motorcycle\'}.get(category))\n                        if cls is not None:\n                            box = Box(ann[\'translation\'], ann[\'size\'], Quaternion(ann[\'rotation\']))\n                            world_boxes.append((box.corners(), CLASSES.index(cls)))\n                    available_channels = set()\n                    for sd in children(\'sample_data\', sample[\'token\']):\n                        cs = calibrated[sd[\'calibrated_sensor_token\']]\n                        channel = sensors[cs[\'sensor_token\']][\'channel\']\n                        if channel not in CAMERAS:\n                            continue\n                        available_channels.add(channel)\n                        image = root / sd[\'filename\']\n                        if not image.is_file():\n                            data[\'missing_images\'].append(str(image))\n                            continue\n                        ego = get(\'ego_pose\', sd[\'ego_pose_token\'])\n                        rot_ego = Quaternion(ego[\'rotation\']).rotation_matrix.T\n                        rot_cam = Quaternion(cs[\'rotation\']).rotation_matrix.T\n                        rotation = rot_cam @ rot_ego\n                        translation = -rotation @ np.array(ego[\'translation\']) - rot_cam @ np.array(cs[\'translation\'])\n                        intrinsic = np.array(cs[\'camera_intrinsic\'])\n                        boxes, labels = [], []\n                        for corners, cls in world_boxes:\n                            camera = rotation @ corners + translation[:, None]\n                            camera = camera[:, camera[2] > 0]\n                            if camera.shape[1] < 3:\n                                continue\n                            projected = intrinsic @ camera\n                            points = (projected[:2] / projected[2]).T\n                            bounds = clipped_bounds(points, sd[\'width\'], sd[\'height\'])\n                            if bounds is not None:\n                                boxes.append(bounds); labels.append(cls)\n                        data[\'rows\'].append({\'id\': f\'nuscenes/{sd["token"]}\', \'image\': str(image),\n                            \'dataset\': \'nuscenes\', \'group\': scene[\'name\'], \'split\': split, \'beta\': None,\n                            \'camera\': channel, \'width\': sd[\'width\'], \'height\': sd[\'height\'],\n                            \'boxes\': boxes, \'labels\': labels, \'iscrowd\': [0]*len(boxes),\n                            \'label_policy\': \'projected 3D boxes, all visibility levels; 2D proxy benchmark\'})\n                    data[\'expected_images\'] += len(CAMERAS)\n                    data[\'missing_channels\'].extend(f\'{sample["token"]}/{c}\' for c in set(CAMERAS)-available_channels)\n                atomic_json(shard, data)\n            report[\'expected_images\'] += data[\'expected_images\']\n            report[\'available_images\'] += len(data[\'rows\'])\n            report[\'missing_images\'].extend(data[\'missing_images\'])\n            report[\'missing_channels\'].extend(data[\'missing_channels\'])\n            shard_paths.append(shard)\n            if len(shard_paths) % 25 == 0:\n                atomic_json(output.with_suffix(\'.coverage.json\'), report)\n        connection.close()\n        complete = (len(ordered) == len(train_names | test_names) and not report[\'missing_images\']\n                    and not report[\'missing_channels\'])\n        report[\'complete\'] = complete\n        atomic_json(output.with_suffix(\'.coverage.json\'), report)\n        if not complete:\n            raise RuntimeError(f\'nuScenes coverage incomplete: {report["available_images"]}/{report["expected_images"]} images. \'\n                               f\'See {output.with_suffix(".coverage.json")}. Download missing images, then rerun; \'\n                               \'completed scenes resume after the remaining sensor archives are extracted.\')\n        temporary = output.with_suffix(\'.json.tmp\')\n        counts = Counter()\n        with temporary.open(\'w\', encoding=\'utf-8\') as handle:\n            handle.write(\'[\')\n            first = True\n            for shard in shard_paths:\n                for row in json.loads(shard.read_text(encoding=\'utf-8\'))[\'rows\']:\n                    if not first:\n                        handle.write(\',\\n\')\n                    handle.write(json.dumps(row)); first = False\n                    counts[row[\'split\']] += 1\n            handle.write(\']\')\n        os.replace(temporary, output)\n        print(f\'Prepared {report["available_images"]} images; complete={complete}; splits={dict(counts)}\', flush=True)\n        return output\n', 'notebooks/_support/fog_research.py': '"""Detector, density-conditioning and scoring primitives for the fog study.\n\nThe supported full training entry point is fog_training.run_training.\n"""\nimport contextlib\nimport hashlib\nimport io\nimport json\nimport random\nimport time\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch import nn\nfrom torch.nn import functional as F\n\nCLASSES = [\'person\', \'car\', \'truck\', \'bus\', \'train\', \'motorcycle\', \'bicycle\']\nCOCO_IDS = [1, 3, 8, 6, 7, 4, 2]\nYOLO_IDS = [0, 2, 7, 5, 6, 3, 1]\nMODEL_SPECS = {\'YOLO11n\': \'yolo11n.pt\', \'YOLO11m\': \'yolo11m.pt\',\n               \'DETR\': \'facebook/detr-resnet-50\'}\nARMS = [\'rgb_control\', \'predicted_density\', \'oracle_density\']\nBETA_SCALE = .02\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef digest(path):\n    result = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef read_manifest(path, verify_images=False):\n    """Fail closed on incomplete datasets; never turn missing annotations into empty GT."""\n    path = Path(path)\n    records = json.loads(path.read_text(encoding=\'utf-8\'))\n    if not isinstance(records, list) or not records:\n        raise ValueError(\'Manifest must contain a nonempty list of labeled images.\')\n    ids, images, groups = set(), set(), {}\n    for row in records:\n        for key in (\'id\', \'image\', \'dataset\', \'split\', \'group\', \'boxes\', \'labels\', \'beta\'):\n            if key not in row:\n                raise ValueError(f\'Missing {key} in manifest row\')\n        if row[\'dataset\'] not in (\'foggy_cityscapes\', \'nuscenes\'):\n            raise ValueError(\'Unknown dataset\')\n        if row[\'split\'] not in (\'density_train\', \'detector_train\', \'val\', \'test\'):\n            raise ValueError(\'Invalid split\')\n        if not row[\'id\'] or not row[\'group\']:\n            raise ValueError(\'Empty image/group identity\')\n        image = (path.parent / row[\'image\']).resolve()\n        if row[\'id\'] in ids or str(image) in images:\n            raise ValueError(\'Duplicate image identity/path\')\n        ids.add(row[\'id\']); images.add(str(image))\n        if not image.is_file():\n            raise FileNotFoundError(image)\n        row[\'image\'] = str(image)\n        if verify_images or \'width\' not in row or \'height\' not in row:\n            with Image.open(image) as img:\n                width, height = img.size\n                if verify_images:\n                    img.verify()\n        else:\n            width, height = row[\'width\'], row[\'height\']\n        if width <= 0 or height <= 0:\n            raise ValueError(\'Invalid image dimensions\')\n        boxes = np.asarray(row[\'boxes\'], dtype=float).reshape(-1, 4)\n        labels = np.asarray(row[\'labels\'])\n        if len(boxes) != len(labels) or not np.isfinite(boxes).all():\n            raise ValueError(\'Invalid boxes/labels\')\n        crowds = row.get(\'iscrowd\', [0] * len(boxes))\n        if len(crowds) != len(boxes) or any(c not in (0, 1) for c in crowds):\n            raise ValueError(\'iscrowd must have one binary flag per annotation\')\n        if len(labels) and (not np.isin(labels, range(len(CLASSES))).all()):\n            raise ValueError(\'Labels must use shared class IDs 0..6\')\n        if len(boxes) and (np.any(boxes[:, 2:] <= boxes[:, :2]) or\n                           np.any(boxes < 0) or np.any(boxes > [width, height, width, height])):\n            raise ValueError(\'Boxes must be positive-area XYXY pixels inside the image\')\n        beta = row[\'beta\']\n        if beta is not None and (not np.isfinite(beta) or not 0 <= beta <= BETA_SCALE):\n            raise ValueError(\'Beta must be null or an attenuation coefficient in [0, .02] m^-1\')\n        if row[\'dataset\'] == \'foggy_cityscapes\' and beta is None:\n            raise ValueError(\'Foggy Cityscapes requires known beta\')\n        key = (row[\'dataset\'], row[\'group\'])\n        if key in groups and groups[key] != row[\'split\']:\n            raise ValueError(\'Scene/sequence leakage across splits\')\n        groups[key] = row[\'split\']\n        row[\'width\'], row[\'height\'] = width, height\n    return records\n\n\ndef select_groups(rows, limit=None):\n    """Select whole sequences, preserving all fog variants; None selects everything."""\n    keys = sorted({(r[\'dataset\'], r[\'group\']) for r in rows})\n    if limit is not None:\n        if limit < 1:\n            raise ValueError(\'Group limit must be positive or None\')\n        keys = keys[:limit]\n    keys = set(keys)\n    return sorted([r for r in rows if (r[\'dataset\'], r[\'group\']) in keys], key=lambda r: r[\'id\'])\n\n\ndef image_tensor(row, size, device):\n    with Image.open(row[\'image\']) as image:\n        image = image.convert(\'RGB\').resize((size, size), Image.Resampling.BILINEAR)\n        array = np.asarray(image).copy()\n    return torch.from_numpy(array).permute(2, 0, 1).float().unsqueeze(0).to(device) / 255\n\n\nclass DensityCNN(nn.Module):\n    """Small scalar regressor trained from scratch; output beta in m^-1."""\n    def __init__(self):\n        super().__init__()\n        self.features = nn.Sequential(nn.Conv2d(3, 16, 5, 2, 2), nn.ReLU(),\n                                      nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),\n                                      nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),\n                                      nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 1))\n\n    def forward(self, images):\n        return self.features(F.interpolate(images, (128, 128), mode=\'bilinear\',\n                                           align_corners=False)).sigmoid() * BETA_SCALE\n\n\nclass FogAdapter(nn.Module):\n    """Identity-initialized RGB residual with scalar FiLM at hidden features.\n\n    The control has exactly the same parameters, but always receives zero.\n    This tests input adaptation, not internal YOLO/DETR feature conditioning.\n    """\n    def __init__(self):\n        super().__init__()\n        self.features = nn.Conv2d(3, 8, 3, padding=1)\n        self.film = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 16))\n        self.output = nn.Conv2d(8, 3, 1)\n        nn.init.zeros_(self.output.weight)\n        nn.init.zeros_(self.output.bias)\n\n    def forward(self, image, beta):\n        scale, shift = self.film(beta.reshape(-1, 1) / BETA_SCALE).chunk(2, dim=1)\n        features = self.features(image)\n        features = features * (1 + scale[:, :, None, None]) + shift[:, :, None, None]\n        return (image + .1 * self.output(F.relu(features)).tanh()).clamp(0, 1)\n\n\ndef train_density(rows, device, epochs, seed):\n    if not rows or any(r[\'split\'] != \'density_train\' or r[\'beta\'] is None for r in rows):\n        raise ValueError(\'Density training requires a separate labeled density_train split\')\n    if len({r[\'beta\'] for r in rows}) < 2:\n        raise ValueError(\'Density estimator needs multiple density values\')\n    if epochs < 1:\n        raise ValueError(\'Training epochs must be positive\')\n    seed_all(seed)\n    model = DensityCNN().to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)\n    history = []\n    for epoch in range(epochs):\n        order = random.Random(seed + epoch).sample(rows, len(rows))\n        losses = []\n        for row in order:\n            x = image_tensor(row, 128, device)\n            target = torch.tensor([[row[\'beta\']]], device=device)\n            optimizer.zero_grad()\n            loss = F.mse_loss(model(x) / BETA_SCALE, target / BETA_SCALE)\n            loss.backward(); optimizer.step()\n            losses.append(float(loss.detach()))\n        history.append({\'epoch\': epoch + 1, \'normalized_mse\': float(np.mean(losses))})\n    return model.eval().requires_grad_(False), history\n\n\nclass Detector:\n    """Differentiable detector loss plus consistent, original-pixel predictions."""\n    def __init__(self, name, checkpoint, device, trainable=True):\n        self.name, self.device = name, device\n        self.yolo = name.startswith(\'YOLO\')\n        if self.yolo:\n            # Ultralytics selects compiled NMS only when torchvision is loaded.\n            # Load it before warm-up/timing to avoid the slow Python fallback.\n            import torchvision.ops\n            from ultralytics import YOLO\n            from ultralytics.utils import DEFAULT_CFG_DICT\n            self.model = YOLO(str(checkpoint)).model.to(device).float()\n            self.model.args = SimpleNamespace(**DEFAULT_CFG_DICT)\n        else:\n            from transformers import DetrConfig, DetrForObjectDetection\n            kwargs = {} if Path(checkpoint).is_dir() else {\'revision\': \'no_timm\'}\n            raw, _ = DetrConfig.get_config_dict(str(checkpoint), **kwargs)\n            # The official no_timm export predates strict bool validation in v5.\n            if raw.get(\'dilation\') is None:\n                raw[\'dilation\'] = False\n            if raw.get(\'use_pretrained_backbone\') is None:\n                raw[\'use_pretrained_backbone\'] = False\n            configuration = DetrConfig.from_dict(raw)\n            self.model = DetrForObjectDetection.from_pretrained(str(checkpoint), config=configuration, **kwargs).to(device)\n        self.model.eval().requires_grad_(trainable)\n        if self.yolo:\n            # YOLO\'s fixed distribution-to-distance integral is not a learned convolution.\n            for name, parameter in self.model.named_parameters():\n                if \'.dfl.\' in name:\n                    parameter.requires_grad_(False)\n\n    def training_mode(self):\n        self.model.train()\n        # Batch-one microsteps: preserve pretrained BN statistics, but train affine weights.\n        for module in self.model.modules():\n            if isinstance(module, nn.modules.batchnorm._BatchNorm):\n                module.eval()\n\n    def forward(self, x):\n        if self.yolo:\n            return self.model(x)\n        mean = x.new_tensor([.485, .456, .406])[None, :, None, None]\n        std = x.new_tensor([.229, .224, .225])[None, :, None, None]\n        return self.model(pixel_values=(x - mean) / std)\n\n    def loss(self, x, row):\n        boxes = x.new_tensor(row[\'boxes\']).reshape(-1, 4)\n        boxes = boxes / x.new_tensor([row[\'width\'], row[\'height\']] * 2)\n        center = (boxes[:, :2] + boxes[:, 2:]) / 2\n        wh = boxes[:, 2:] - boxes[:, :2]\n        boxes = torch.cat([center, wh], dim=1)\n        mapping = YOLO_IDS if self.yolo else COCO_IDS\n        labels = torch.tensor([mapping[i] for i in row[\'labels\']], device=self.device)\n        if self.yolo:\n            batch = {\'img\': x, \'batch_idx\': x.new_zeros(len(labels)),\n                     \'cls\': labels.reshape(-1, 1), \'bboxes\': boxes}\n            return self.model.loss(batch, self.forward(x))[0].sum()\n        mean = x.new_tensor([.485, .456, .406])[None, :, None, None]\n        std = x.new_tensor([.229, .224, .225])[None, :, None, None]\n        return self.model(pixel_values=(x - mean) / std,\n                          labels=[{\'class_labels\': labels.long(), \'boxes\': boxes}]).loss\n\n    def predict(self, x, row):\n        output = self.forward(x)\n        if self.yolo:\n            from ultralytics.utils.nms import non_max_suppression\n            result = non_max_suppression(output, conf_thres=.001, iou_thres=.7,\n                                         classes=YOLO_IDS, max_det=100)[0]\n            boxes, scores, labels = result[:, :4], result[:, 4], result[:, 5].long()\n            boxes = boxes / x.shape[-1]\n            mapping = YOLO_IDS\n        else:\n            probabilities = output.logits.softmax(-1)[0, :, :-1]\n            scores, labels = probabilities.max(-1)\n            cxcy, wh = output.pred_boxes[0].split(2, dim=-1)\n            boxes = torch.cat([cxcy - wh / 2, cxcy + wh / 2], dim=-1)\n            mapping = COCO_IDS\n        boxes = boxes.clamp(0, 1) * x.new_tensor([row[\'width\'], row[\'height\']] * 2)\n        result = []\n        for box, score, label in zip(boxes.detach().cpu().tolist(), scores.detach().cpu().tolist(),\n                                     labels.detach().cpu().tolist()):\n            if label in mapping and score >= .001 and box[2] > box[0] and box[3] > box[1]:\n                result.append({\'box\': box, \'score\': score, \'label\': mapping.index(label)})\n        return sorted(result, key=lambda r: -r[\'score\'])[:100]\n\n\ndef beta_for(arm, x, row, density):\n    if arm == \'rgb_control\':\n        return x.new_zeros((1, 1))\n    if arm == \'predicted_density\':\n        if density is None:\n            raise ValueError(\'A trained density estimator is required\')\n        with torch.no_grad():\n            return density(x)\n    if arm == \'oracle_density\' and row[\'beta\'] is not None:\n        return x.new_tensor([[row[\'beta\']]])\n    raise ValueError(\'Oracle density is unavailable; do not substitute zero or a prediction\')\n\n\ndef train_adapter(detector, rows, density, arm, device, size, epochs, seed):\n    if not rows or any(r[\'split\'] != \'detector_train\' for r in rows) or epochs < 1:\n        raise ValueError(\'Adapter training requires detector_train rows and positive epochs\')\n    seed_all(seed)\n    adapter = FogAdapter().to(device)\n    optimizer = torch.optim.Adam(adapter.parameters(), lr=1e-3)\n    history = []\n    for epoch in range(epochs):\n        losses = []\n        for row in random.Random(seed + epoch).sample(rows, len(rows)):\n            x = image_tensor(row, size, device)\n            beta = beta_for(arm, x, row, density)\n            optimizer.zero_grad()\n            loss = detector.loss(adapter(x, beta), row)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite detector loss\')\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(adapter.parameters(), 10)\n            optimizer.step()\n            losses.append(float(loss.detach()))\n        history.append({\'epoch\': epoch + 1, \'loss\': float(np.mean(losses))})\n    return adapter.eval(), history\n\n\ndef evaluate(detector, adapter, density, arm, rows, device, size, repeats=3):\n    if not rows or repeats < 1:\n        raise ValueError(\'Evaluation requires images and positive repeats\')\n    def sync():\n        if str(device).startswith(\'cuda\'):\n            torch.cuda.synchronize()\n    @torch.inference_mode()\n    def single(row):\n        # Start before decode, resize, transfer and scalar estimation.\n        sync(); start = time.perf_counter()\n        x = image_tensor(row, size, device)\n        sync(); prepared = time.perf_counter()\n        beta = beta_for(arm, x, row, density)\n        sync(); estimated = time.perf_counter()\n        prediction = detector.predict(adapter(x, beta), row)\n        sync(); finished = time.perf_counter()\n        return prediction, float(beta.item()), {\n            \'total_ms\': (finished - start) * 1000,\n            \'density_ms\': (estimated - prepared) * 1000,\n            \'detector_adapter_ms\': (finished - estimated) * 1000}\n    for _ in range(3):\n        single(rows[0])\n    results = []\n    for row in rows:\n        timings = []\n        for _ in range(repeats):\n            prediction, beta, timing = single(row)\n            timings.append(timing)\n        results.append({\'id\': row[\'id\'], \'prediction\': prediction, \'beta_input\': beta,\n                        \'timings\': timings})\n    return results\n\n\ndef coco_metrics(rows, results):\n    """COCO bbox AP, including crowd annotations; seven shared categories."""\n    from pycocotools.coco import COCO\n    from pycocotools.cocoeval import COCOeval\n    if [r[\'id\'] for r in rows] != [r[\'id\'] for r in results]:\n        raise ValueError(\'Predictions are not paired with the exact evaluation images\')\n    images, annotations, predictions = [], [], []\n    for index, (row, result) in enumerate(zip(rows, results), 1):\n        images.append({\'id\': index, \'width\': row[\'width\'], \'height\': row[\'height\']})\n        crowds = row.get(\'iscrowd\', [0] * len(row[\'boxes\']))\n        for box, label, crowd in zip(row[\'boxes\'], row[\'labels\'], crowds):\n            x1, y1, x2, y2 = box\n            annotations.append({\'id\': len(annotations) + 1, \'image_id\': index,\n                                \'category_id\': int(label), \'bbox\': [x1, y1, x2-x1, y2-y1],\n                                \'area\': (x2-x1)*(y2-y1), \'iscrowd\': int(crowd)})\n        for pred in result[\'prediction\']:\n            x1, y1, x2, y2 = pred[\'box\']\n            predictions.append({\'image_id\': index, \'category_id\': pred[\'label\'],\n                                \'bbox\': [x1, y1, x2-x1, y2-y1], \'score\': pred[\'score\']})\n    truth = COCO()\n    truth.dataset = {\'info\': {}, \'images\': images, \'annotations\': annotations,\n                     \'categories\': [{\'id\': i, \'name\': n} for i, n in enumerate(CLASSES)]}\n    with contextlib.redirect_stdout(io.StringIO()):\n        truth.createIndex()\n        if predictions:\n            predicted = truth.loadRes(predictions)\n        else:\n            predicted = COCO()\n            predicted.dataset = {**truth.dataset, \'annotations\': []}\n            predicted.createIndex()\n        evaluator = COCOeval(truth, predicted, \'bbox\')\n        evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()\n    tp = fp = positives = 0\n    for item in evaluator.evalImgs:\n        if item is None or item[\'aRng\'] != evaluator.params.areaRng[0]:\n            continue\n        keep = np.asarray(item[\'dtScores\']) >= .25\n        ignored = np.asarray(item[\'dtIgnore\'])[0].astype(bool)\n        matched = np.asarray(item[\'dtMatches\'])[0] > 0\n        tp += int(np.sum(keep & ~ignored & matched))\n        fp += int(np.sum(keep & ~ignored & ~matched))\n        positives += int(np.sum(~np.asarray(item[\'gtIgnore\']).astype(bool)))\n    metrics = {\'mAP50_95\': float(evaluator.stats[0]), \'AP50\': float(evaluator.stats[1]),\n               \'AR100\': float(evaluator.stats[8]), \'precision_at_025\': tp/(tp+fp) if tp+fp else 0.,\n               \'recall_at_025\': tp/positives if positives else np.nan}\n    for index, name in enumerate(CLASSES):\n        values = evaluator.eval[\'precision\'][:, :, index, 0, -1]\n        supported = values[values >= 0]\n        metrics[f\'AP_{name}\'] = float(supported.mean()) if len(supported) else np.nan\n    return {key: np.nan if value == -1 else value for key, value in metrics.items()}\n\n\ndef summarize(rows, results):\n    groups = [(\'all\', list(range(len(rows))))]\n    for beta in sorted({r[\'beta\'] for r in rows if r[\'beta\'] is not None}):\n        groups.append((str(beta), [i for i, r in enumerate(rows) if r[\'beta\'] == beta]))\n    if any(r[\'beta\'] is None for r in rows):\n        groups.append((\'unknown\', [i for i, r in enumerate(rows) if r[\'beta\'] is None]))\n    summaries = []\n    for bucket, indices in groups:\n        subset = [rows[i] for i in indices]\n        outputs = [results[i] for i in indices]\n        values = [t[\'total_ms\'] for r in outputs for t in r[\'timings\']]\n        metric = coco_metrics(subset, outputs)\n        summaries.append({\'density\': bucket, \'images\': len(subset), **metric,\n                          \'mean_ms\': float(np.mean(values)), \'p50_ms\': float(np.median(values)),\n                          \'p95_ms\': float(np.percentile(values, 95)),\n                          \'serial_fps\': 1000 / float(np.mean(values)),\n                          \'density_mean_ms\': float(np.mean([t[\'density_ms\'] for r in outputs for t in r[\'timings\']]))})\n    return summaries\n\n\ndef plot_comparison(results):\n    import matplotlib.pyplot as plt\n    if results.empty:\n        print(\'No measured results. Prepare data and run experiments; no placeholder scores are plotted.\')\n        return\n    for dataset, frame in results.groupby(\'dataset\'):\n        overall = frame[frame.density == \'all\']\n        fig, axes = plt.subplots(1, 2, figsize=(13, 5))\n        markers = {\'rgb_control\': \'o\', \'predicted_density\': \'^\', \'oracle_density\': \'s\'}\n        for (model, arm), part in overall.groupby([\'model\', \'arm\']):\n            axes[0].scatter(part.mean_ms, part.mAP50_95, marker=markers[arm], label=f\'{model}: {arm}\')\n        axes[0].set(xlabel=\'End-to-end mean latency (ms)\', ylabel=\'COCO bbox mAP@[.50:.95]\',\n                    title=f\'{dataset}: accuracy vs latency\', ylim=(0, 1))\n        numeric = frame[~frame.density.isin([\'all\', \'unknown\'])].copy()\n        numeric[\'beta\'] = numeric.density.astype(float)\n        for (model, arm), part in numeric.groupby([\'model\', \'arm\']):\n            means = part.groupby(\'beta\').mAP50_95.mean()\n            axes[1].plot(means.index, means.values, marker=markers[arm], label=f\'{model}: {arm}\')\n        axes[1].set(xlabel=\'Known attenuation coefficient beta (m^-1)\', ylabel=\'Mean mAP across seeds\',\n                    title=\'Accuracy by known fog density\', ylim=(0, 1))\n        for axis in axes:\n            axis.grid(alpha=.2)\n        axes[0].legend(fontsize=7, loc=\'best\')\n        fig.tight_layout(); plt.show()\n\n\ndef _legacy_adapter_experiment(rows, output, checkpoints, device=\'cpu\', size=640, seeds=(42, 43, 44),\n                   density_epochs=10, adapter_epochs=5, repeats=3):\n    """Sequential runs; immutable output directory prevents stale/concurrent result mixing."""\n    import importlib.metadata\n    if size < 32 or size % 32:\n        raise ValueError(\'Image size must be a positive multiple of 32\')\n    if not seeds or len(set(seeds)) != len(seeds):\n        raise ValueError(\'Use unique nonempty seeds\')\n    density_rows = [r for r in rows if r[\'split\'] == \'density_train\']\n    # Excluding crowd-bearing training images avoids treating ignored objects as negatives.\n    train_rows = [r for r in rows if r[\'split\'] == \'detector_train\' and not any(r.get(\'iscrowd\', []))]\n    test_sets = {d: [r for r in rows if r[\'split\'] == \'test\' and r[\'dataset\'] == d]\n                 for d in sorted({r[\'dataset\'] for r in rows})}\n    test_sets = {d: r for d, r in test_sets.items() if r}\n    if not density_rows or not train_rows or not test_sets:\n        raise ValueError(\'Need independent density_train, detector_train and test data; prepare Foggy Cityscapes first\')\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    settings = {\'size\': size, \'seeds\': list(seeds), \'density_epochs\': density_epochs,\n                \'adapter_epochs\': adapter_epochs, \'repeats\': repeats, \'device\': str(device),\n                \'checkpoints\': {k: str(v) for k, v in checkpoints.items()},\n                \'method\': \'frozen detector, trained input FiLM adapter; identical zero-density control\',\n                \'versions\': {p: importlib.metadata.version(p) for p in\n                             (\'torch\', \'transformers\', \'ultralytics\', \'pycocotools\')},\n                \'hardware\': torch.cuda.get_device_name() if str(device).startswith(\'cuda\') else \'CPU\',\n                \'excluded_crowd_train_images\': sum(r[\'split\'] == \'detector_train\' and any(r.get(\'iscrowd\', [])) for r in rows),\n                \'input_sha256\': {r[\'id\']: digest(r[\'image\']) for r in rows},\n                \'source_sha256\': digest(__file__)}\n    (output / \'settings.json\').write_text(json.dumps(settings, indent=2), encoding=\'utf-8\')\n    (output / \'manifest.json\').write_text(json.dumps(rows, indent=2), encoding=\'utf-8\')\n    summaries, status, density_scores = [], [], []\n    for seed in seeds:\n        density, history = train_density(density_rows, device, density_epochs, seed)\n        torch.save(density.state_dict(), output / f\'density-{seed}.pt\')\n        pd.DataFrame(history).to_csv(output / f\'density-{seed}-training.csv\', index=False)\n        # Held-out density quality, split by dataset; null beta is never scored.\n        for dataset, subset in test_sets.items():\n            known = [r for r in subset if r[\'beta\'] is not None]\n            with torch.inference_mode():\n                estimates = [float(density(image_tensor(r, size, device)).item()) for r in known]\n            if known:\n                error = np.array(estimates) - np.array([r[\'beta\'] for r in known])\n                density_scores.append({\'seed\': seed, \'dataset\': dataset, \'images\': len(known),\n                                       \'MAE_beta\': float(np.abs(error).mean()),\n                                       \'RMSE_beta\': float(np.sqrt((error**2).mean()))})\n        pd.DataFrame(density_scores).to_csv(output / \'density-quality.csv\', index=False)\n        for name, checkpoint in checkpoints.items():\n            detector = None\n            try:\n                detector = Detector(name, checkpoint, device)\n                # Hash the actual loaded parameters, including downloaded HF weights.\n                weights_hash = hashlib.sha256()\n                for key, tensor in detector.model.state_dict().items():\n                    weights_hash.update(key.encode())\n                    weights_hash.update(tensor.detach().cpu().contiguous().numpy().tobytes())\n                (output / f\'{name}-{seed}-weights.sha256\').write_text(weights_hash.hexdigest())\n                for arm in ARMS:\n                    adapter, history = train_adapter(detector, train_rows, density, arm,\n                                                     device, size, adapter_epochs, seed)\n                    torch.save(adapter.state_dict(), output / f\'{name}-{seed}-{arm}.pt\')\n                    pd.DataFrame(history).to_csv(output / f\'{name}-{seed}-{arm}-training.csv\', index=False)\n                    for dataset, subset in test_sets.items():\n                        identity = {\'model\': name, \'arm\': arm, \'seed\': seed, \'dataset\': dataset}\n                        if arm == \'oracle_density\' and any(r[\'beta\'] is None for r in subset):\n                            status.append({**identity, \'status\': \'unavailable\', \'reason\': \'No known beta for this dataset\'})\n                            continue\n                        if str(device).startswith(\'cuda\'):\n                            torch.cuda.reset_peak_memory_stats()\n                        predictions = evaluate(detector, adapter, density, arm, subset, device, size, repeats)\n                        (output / f\'{name}-{seed}-{arm}-{dataset}.json\').write_text(\n                            json.dumps(predictions), encoding=\'utf-8\')\n                        peak = torch.cuda.max_memory_allocated() / 2**20 if str(device).startswith(\'cuda\') else None\n                        for summary in summarize(subset, predictions):\n                            summaries.append({**identity, **summary, \'peak_allocated_gpu_mb\': peak})\n                        status.append({**identity, \'status\': \'complete\'})\n                        pd.DataFrame(summaries).to_csv(output / \'results.csv\', index=False)\n                        pd.DataFrame(status).to_csv(output / \'status.csv\', index=False)\n            except Exception as error:\n                status.append({\'model\': name, \'seed\': seed, \'status\': \'error\',\n                               \'reason\': f\'{type(error).__name__}: {error}\'})\n                print(f\'{name}, seed {seed}: {error}\')\n            finally:\n                del detector\n                if torch.cuda.is_available():\n                    torch.cuda.empty_cache()\n                pd.DataFrame(status).to_csv(output / \'status.csv\', index=False)\n    return pd.DataFrame(summaries), pd.DataFrame(status)\n', 'notebooks/_support/fog_training.py': '"""Full detector fine-tuning, resumable checkpoints, preflight, and persistent evaluation."""\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom filelock import FileLock\nfrom tqdm.auto import tqdm\n\nfrom fog_prepare import atomic_json\nfrom fog_research import (BETA_SCALE, CLASSES, DensityCNN, Detector, FogAdapter, beta_for,\n                          digest, image_tensor, seed_all, summarize)\n\nENGINE_VERSION = 4\n\n\ndef default_config(root):\n    root = Path(root).resolve()\n    return {\'output\': str(root/\'reports/fog/full-training-complete\'),\n            \'manifest\': str(root/\'data/fog/nuscenes_all.json\'),\n            \'city_manifest\': str(root/\'data/fog/cityscapes.json\'),\n            \'density_checkpoint\': None,\n            \'checkpoints\': {\'YOLO11n\': str(root/\'yolo11n.pt\'), \'YOLO11m\': str(root/\'yolo11m.pt\'),\n                            \'DETR\': \'facebook/detr-resnet-50\'},\n            \'arms\': [\'rgb_control\', \'predicted_density\'], \'seeds\': [42, 43, 44],\n            \'device\': \'cuda:0\' if torch.cuda.is_available() else \'cpu\',\n            \'size\': 640, \'epochs\': 30, \'density_epochs\': 15,\n            \'accumulate\': 4, \'save_every\': 200, \'repeats\': 1,\n            \'learning_rates\': {\'YOLO11n\': 1e-4, \'YOLO11m\': 1e-4, \'DETR\': 1e-5},\n            \'adapter_lr\': 1e-3, \'amp\': True}\n\n\ndef fingerprint(rows):\n    # Annotation/identity fingerprint plus image stat records; does not re-read every JPEG.\n    value = hashlib.sha256()\n    for row in sorted(rows, key=lambda r: r[\'id\']):\n        stat = Path(row[\'image\']).stat()\n        value.update(json.dumps([row, stat.st_size, stat.st_mtime_ns], sort_keys=True).encode())\n    return value.hexdigest()\n\n\ndef partitions(rows):\n    parts = {name: [r for r in rows if r[\'split\'] == name and\n                   (name == \'density_train\' or r.get(\'detection_labeled\', True)) and\n                   (name not in (\'detector_train\', \'val\') or not any(r.get(\'iscrowd\', [])))]\n             for name in (\'detector_train\', \'density_train\', \'val\', \'test\')}\n    parts[\'density_val\'] = [r for r in rows if r[\'split\'] == \'val\' and r[\'beta\'] is not None]\n    parts[\'density_test\'] = [r for r in rows if r[\'split\'] == \'test\' and r[\'beta\'] is not None]\n    return parts\n\n\ndef preflight(config, rows):\n    """Return actionable checks without starting training or downloading anything."""\n    parts = partitions(rows)\n    checks = []\n    def check(name, good, detail):\n        checks.append({\'check\': name, \'ready\': bool(good), \'detail\': detail})\n    check(\'training images\', bool(parts[\'detector_train\']), f\'{len(parts["detector_train"])} labeled images\')\n    check(\'tuning images\', bool(parts[\'val\']), f\'{len(parts["val"])} scene-held-out images for checkpoint selection\')\n    check(\'test images\', bool(parts[\'test\']), f\'{len(parts["test"])} held-out images for final metrics\')\n    valid_arms = {\'rgb_control\', \'predicted_density\', \'oracle_density\'}\n    check(\'arms\', bool(config[\'arms\']) and len(set(config[\'arms\'])) == len(config[\'arms\']) and\n          set(config[\'arms\']) <= valid_arms, \', \'.join(config[\'arms\']))\n    groups = {}\n    clean = True\n    ids = set()\n    for row in rows:\n        key = (row[\'dataset\'], row[\'group\'])\n        clean &= key not in groups or groups[key] == row[\'split\']\n        clean &= row[\'id\'] not in ids\n        groups[key] = row[\'split\']; ids.add(row[\'id\'])\n    check(\'scene/ID isolation\', clean, \'No scene crosses train, tuning, density-training or final-test splits\')\n    for key in (\'size\', \'epochs\', \'density_epochs\', \'accumulate\', \'save_every\', \'repeats\'):\n        check(key, isinstance(config[key], int) and config[key] > 0, str(config[key]))\n    check(\'size alignment\', config[\'size\'] % 32 == 0, \'Image size must be divisible by 32\')\n    check(\'seeds\', bool(config[\'seeds\']) and len(set(config[\'seeds\'])) == len(config[\'seeds\']), str(config[\'seeds\']))\n    coverage_path = Path(config[\'manifest\']).with_suffix(\'.coverage.json\')\n    if any(r[\'dataset\'] == \'nuscenes\' for r in rows):\n        coverage = json.loads(coverage_path.read_text()) if coverage_path.exists() else {}\n        complete = coverage.get(\'complete\') is True\n        check(\'nuScenes coverage\', complete,\n              f\'complete={complete}; coverage report: {coverage_path}. All expected camera keyframes are required.\')\n    if \'predicted_density\' in config[\'arms\']:\n        source = config.get(\'density_checkpoint\')\n        if source:\n            try:\n                saved = torch.load(source, map_location=\'cpu\', weights_only=True)\n                correct = saved.get(\'kind\') == \'fog-density-v1\' and saved.get(\'beta_scale\') == BETA_SCALE\n                trained_groups = {tuple(x) for x in saved.get(\'train_groups\', [])}\n                held_out = {(r[\'dataset\'], r[\'group\']) for r in rows if r[\'split\'] in (\'val\', \'test\', \'detector_train\')}\n                correct &= bool(trained_groups) and not (trained_groups & held_out)\n                check(\'density checkpoint\', correct, \'Requires this pipeline format, beta units, and independent training groups\')\n            except Exception as error:\n                check(\'density checkpoint\', False, f\'{type(error).__name__}: {error}\')\n        else:\n            train = parts[\'density_train\']\n            known_val = parts[\'density_val\']\n            check(\'density supervision\', bool(train) and all(r[\'beta\'] is not None for r in train)\n                  and len({r[\'beta\'] for r in train}) > 1 and bool(known_val),\n                  \'Add Foggy Cityscapes images (beta comes from filenames; gtFine is not required), prepare its manifest, \'\n                  \'or set density_checkpoint to an independently trained fog-density checkpoint. \'\n                  \'Native nuScenes has no beta labels; its visibility field is NOT fog density.\')\n    if \'oracle_density\' in config[\'arms\']:\n        check(\'oracle labels\', all(r[\'beta\'] is not None for r in parts[\'detector_train\']+parts[\'val\']+parts[\'test\']),\n              \'Oracle is optional and requires known beta for every matched train/validation/test image. \'\n              \'Use the default rgb_control/predicted_density comparison for native nuScenes.\')\n    check(\'models\', bool(config[\'checkpoints\']) and set(config[\'checkpoints\']) <= {\'YOLO11n\', \'YOLO11m\', \'DETR\'},\n          \'Three supported pretrained architectures; missing public weights download on first load\')\n    check(\'device\', not config[\'device\'].startswith(\'cuda\') or torch.cuda.is_available(), config[\'device\'])\n    return pd.DataFrame(checks)\n\n\ndef save_torch(path, payload):\n    temporary = path.with_suffix(\'.tmp\')\n    torch.save(payload, temporary)\n    os.replace(temporary, path)\n\n\ndef fit_density(parts, config, directory, seed):\n    """Train or resume a scalar estimator; select the lowest tuning loss checkpoint."""\n    directory.mkdir(parents=True, exist_ok=True)\n    seed_all(seed)\n    model = DensityCNN().to(config[\'device\'])\n    source = config.get(\'density_checkpoint\')\n    if source:\n        saved = torch.load(source, map_location=config[\'device\'], weights_only=True)\n        model.load_state_dict(saved[\'model\'])\n        return model.eval().requires_grad_(False)\n    train = parts[\'density_train\']\n    val = parts[\'density_val\']\n    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)\n    last, best = directory/\'last.pt\', directory/\'best.pt\'\n    start_epoch, start_step, best_loss = 0, 0, float(\'inf\')\n    running_sum, running_count, history = 0., 0, []\n    if last.exists():\n        state = torch.load(last, map_location=config[\'device\'], weights_only=True)\n        model.load_state_dict(state[\'model\']); optimizer.load_state_dict(state[\'optimizer\'])\n        start_epoch, start_step, best_loss = state[\'epoch\'], state[\'step\'], state[\'best_loss\']\n        running_sum, running_count, history = state[\'running_sum\'], state[\'running_count\'], state[\'history\']\n    def save(epoch, step, improved=False):\n        payload = {\'kind\': \'fog-density-v1\', \'beta_scale\': BETA_SCALE, \'model\': model.state_dict(),\n                   \'optimizer\': optimizer.state_dict(), \'epoch\': epoch, \'step\': step, \'best_loss\': best_loss,\n                   \'running_sum\': running_sum, \'running_count\': running_count, \'history\': history,\n                   \'train_groups\': sorted({(r[\'dataset\'], r[\'group\']) for r in train})}\n        if improved:\n            save_torch(best, payload)\n        save_torch(last, payload)\n        return payload\n    for epoch in range(start_epoch, config[\'density_epochs\']):\n        order = random.Random(seed+epoch).sample(range(len(train)), len(train))\n        model.train()\n        for step in tqdm(range(start_step, len(order)), total=len(order), initial=start_step,\n                         desc=f\'Density seed={seed} epoch={epoch+1}\', mininterval=2):\n            row = train[order[step]]\n            x = image_tensor(row, config[\'size\'], config[\'device\'])\n            target = x.new_tensor([[row[\'beta\']]])\n            optimizer.zero_grad(set_to_none=True)\n            loss = torch.nn.functional.mse_loss(model(x)/BETA_SCALE, target/BETA_SCALE)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite density loss\')\n            loss.backward(); optimizer.step()\n            running_sum += float(loss.detach()); running_count += 1\n            if (step+1) % config[\'save_every\'] == 0:\n                save(epoch, step+1)\n        model.eval()\n        errors = []\n        with torch.inference_mode():\n            for row in tqdm(val, desc=\'Density tuning\', mininterval=2):\n                prediction = model(image_tensor(row, config[\'size\'], config[\'device\'])).item()\n                errors.append((prediction-row[\'beta\'])**2)\n        val_loss = float(np.mean(errors))\n        history.append({\'epoch\': epoch+1, \'train_normalized_mse\': running_sum/running_count,\n                        \'val_beta_rmse\': val_loss**.5})\n        improved = val_loss < best_loss\n        best_loss = min(best_loss, val_loss)\n        running_sum, running_count, start_step = 0., 0, 0\n        save(epoch+1, 0, improved)\n        pd.DataFrame(history).to_csv(directory/\'history.csv\', index=False)\n    model.load_state_dict(torch.load(best, map_location=config[\'device\'], weights_only=True)[\'model\'])\n    return model.eval().requires_grad_(False)\n\n\ndef fit_detector(detector, adapter, density, parts, arm, config, directory, seed):\n    """Train detector AND adapter, with atomic optimizer-boundary resume snapshots."""\n    train, val = parts[\'detector_train\'], parts[\'val\']\n    parameters = [p for p in detector.model.parameters() if p.requires_grad]\n    optimizer = torch.optim.AdamW([{\'params\': parameters, \'lr\': config[\'learning_rates\'][detector.name]},\n                                   {\'params\': adapter.parameters(), \'lr\': config[\'adapter_lr\']}], weight_decay=1e-4)\n    use_amp = config[\'amp\'] and config[\'device\'].startswith(\'cuda\')\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=use_amp, init_scale=128.)\n    last, best = directory/\'last.pt\', directory/\'best.pt\'\n    start_epoch, start_step, best_loss = 0, 0, float(\'inf\')\n    running_sum, running_count, history = 0., 0, []\n    if last.exists():\n        state = torch.load(last, map_location=config[\'device\'], weights_only=True)\n        detector.model.load_state_dict(state[\'model\']); adapter.load_state_dict(state[\'adapter\'])\n        optimizer.load_state_dict(state[\'optimizer\']); scaler.load_state_dict(state[\'scaler\'])\n        start_epoch, start_step, best_loss = state[\'epoch\'], state[\'step\'], state[\'best_loss\']\n        running_sum, running_count, history = state[\'running_sum\'], state[\'running_count\'], state[\'history\']\n        torch.set_rng_state(state[\'rng\'].cpu())\n        if config[\'device\'].startswith(\'cuda\'):\n            torch.cuda.set_rng_state_all([r.cpu() for r in state[\'cuda_rng\']])\n    def snapshot(epoch, step, improved=False, reason=\'periodic\'):\n        payload = {\'kind\': \'fog-detector-v1\', \'model\': detector.model.state_dict(), \'adapter\': adapter.state_dict(),\n                   \'optimizer\': optimizer.state_dict(), \'scaler\': scaler.state_dict(), \'epoch\': epoch, \'step\': step,\n                   \'best_loss\': best_loss, \'history\': history, \'running_sum\': running_sum, \'running_count\': running_count,\n                   \'rng\': torch.get_rng_state(), \'cuda_rng\': torch.cuda.get_rng_state_all() if config[\'device\'].startswith(\'cuda\') else [],\n                   \'checkpoint_reason\': reason}\n        if improved:\n            save_torch(best, payload)\n        save_torch(last, payload)\n        atomic_json(directory/\'progress.json\', {\'model\': detector.name, \'arm\': arm, \'seed\': seed,\n                     \'epoch\': epoch, \'step\': step,\n                     \'best_loss\': float(best_loss) if np.isfinite(best_loss) else None,\n                     \'reason\': reason})\n        return payload\n    def loss_for(x, row, beta):\n        with torch.autocast(device_type=x.device.type, enabled=use_amp):\n            loss = detector.loss(adapter(x, beta), row)\n        if torch.isfinite(loss):\n            return loss\n        if use_amp:\n            # Ultralytics loss can overflow in float16 on a rare large-box\n            # example even when the same update is finite in float32.\n            with torch.autocast(device_type=x.device.type, enabled=False):\n                loss = detector.loss(adapter(x.float(), beta.float()), row)\n            if torch.isfinite(loss):\n                return loss\n        raise ValueError(f\'Nonfinite loss for {row["id"]}; AMP fallback was also nonfinite\')\n    for epoch in range(start_epoch, config[\'epochs\']):\n        detector.training_mode(); adapter.train()\n        order = random.Random(seed+epoch).sample(range(len(train)), len(train))\n        optimizer.zero_grad(set_to_none=True)\n        checkpoint_step = start_step\n        try:\n            for step in tqdm(range(start_step, len(order)), total=len(order), initial=start_step,\n                             desc=f\'{detector.name}/{arm} seed={seed} epoch={epoch+1}\', mininterval=2):\n                row = train[order[step]]\n                x = image_tensor(row, config[\'size\'], config[\'device\'])\n                beta = beta_for(arm, x, row, density)\n                # Tail microbatch uses its actual size instead of downweighting the last examples.\n                window_start = (step//config[\'accumulate\'])*config[\'accumulate\']\n                divisor = min(config[\'accumulate\'], len(order)-window_start)\n                loss = loss_for(x, row, beta)\n                scaler.scale(loss/divisor).backward()\n                running_sum += float(loss.detach()); running_count += 1\n                boundary = (step+1) % config[\'accumulate\'] == 0 or step+1 == len(order)\n                if boundary:\n                    scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(parameters+list(adapter.parameters()), 10)\n                    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)\n                    checkpoint_step = step+1\n                    if checkpoint_step//config[\'accumulate\'] % config[\'save_every\'] == 0 or checkpoint_step == len(order):\n                        snapshot(epoch, checkpoint_step)\n        except KeyboardInterrupt:\n            # Preserve the last completed optimizer boundary. A restart reruns at\n            # most the current accumulation window, without losing model updates.\n            optimizer.zero_grad(set_to_none=True)\n            snapshot(epoch, checkpoint_step, reason=\'interrupt\')\n            raise\n        detector.model.eval(); adapter.eval()\n        validation_losses = []\n        with torch.no_grad():\n            for row in tqdm(val, desc=f\'{detector.name}/{arm} tuning\', mininterval=2):\n                x = image_tensor(row, config[\'size\'], config[\'device\'])\n                loss = loss_for(x, row, beta_for(arm, x, row, density))\n                validation_losses.append(float(loss))\n        val_loss = float(np.mean(validation_losses))\n        if not np.isfinite(val_loss):\n            raise ValueError(\'Nonfinite validation loss\')\n        history.append({\'epoch\': epoch+1, \'train_loss\': running_sum/running_count, \'val_loss\': val_loss})\n        improved = val_loss < best_loss\n        best_loss = min(best_loss, val_loss)\n        running_sum, running_count, start_step = 0., 0, 0\n        snapshot(epoch+1, 0, improved, reason=\'epoch\')\n        pd.DataFrame(history).to_csv(directory/\'history.csv\', index=False)\n    saved = torch.load(best, map_location=config[\'device\'], weights_only=True)\n    detector.model.load_state_dict(saved[\'model\']); adapter.load_state_dict(saved[\'adapter\'])\n    detector.model.eval(); adapter.eval()\n\n\ndef evaluate_resumable(detector, adapter, density, arm, rows, config, path):\n    import time\n    completed = []\n    if path.exists():\n        # A trailing partial line after power loss is removed; earlier lines must be valid.\n        with path.open(\'rb+\') as handle:\n            end = 0\n            while line := handle.readline():\n                if not line.endswith(b\'\\n\'):\n                    handle.truncate(end); break\n                completed.append(json.loads(line)); end = handle.tell()\n        if [r[\'id\'] for r in completed] != [r[\'id\'] for r in rows[:len(completed)]]:\n            raise ValueError(\'Saved predictions do not match input order; use a new run directory\')\n    def sync():\n        if config[\'device\'].startswith(\'cuda\'):\n            torch.cuda.synchronize()\n    @torch.inference_mode()\n    def predict(row):\n        sync(); start = time.perf_counter()\n        x = image_tensor(row, config[\'size\'], config[\'device\'])\n        sync(); before_density = time.perf_counter()\n        beta = beta_for(arm, x, row, density)\n        sync(); after_density = time.perf_counter()\n        predictions = detector.predict(adapter(x, beta), row)\n        sync(); end = time.perf_counter()\n        return predictions, float(beta.item()), {\'total_ms\': (end-start)*1000,\n            \'density_ms\': (after_density-before_density)*1000,\n            \'detector_adapter_ms\': (end-after_density)*1000}\n    if len(completed) < len(rows):\n        for _ in range(3):\n            predict(rows[len(completed)])\n    with path.open(\'a\', encoding=\'utf-8\') as handle:\n        for row in tqdm(rows[len(completed):], desc=f\'Evaluate {detector.name}/{arm}\', mininterval=2):\n            timings = []\n            for _ in range(config[\'repeats\']):\n                prediction, beta, timing = predict(row)\n                timings.append(timing)\n            result = {\'id\': row[\'id\'], \'prediction\': prediction, \'beta_input\': beta, \'timings\': timings}\n            handle.write(json.dumps(result)+\'\\n\'); handle.flush()\n            completed.append(result)\n    return completed\n\n\ndef export_trained_model(detector, adapter, directory, identity, config):\n    """Write a compact inference artifact without optimizer, scaler or RNG state."""\n    payload = {\'kind\': \'fog-detector-export-v1\', **identity,\n               \'checkpoint_source\': config[\'checkpoints\'][detector.name],\n               \'input_size\': config[\'size\'], \'classes\': list(CLASSES),\n               \'model\': {key: value.detach().cpu() for key, value in detector.model.state_dict().items()},\n               \'adapter\': {key: value.detach().cpu() for key, value in adapter.state_dict().items()}}\n    save_torch(directory/\'trained-model.pt\', payload)\n\n\ndef export_saved_checkpoint(name, directory, identity, config):\n    """Create the model-only artifact for a run completed by an older engine."""\n    checkpoint = directory/\'best.pt\'\n    if not checkpoint.exists() or (directory/\'trained-model.pt\').exists():\n        return\n    detector = Detector(name, config[\'checkpoints\'][name], config[\'device\'], trainable=False)\n    adapter = FogAdapter().to(config[\'device\'])\n    saved = torch.load(checkpoint, map_location=config[\'device\'], weights_only=True)\n    detector.model.load_state_dict(saved[\'model\']); adapter.load_state_dict(saved[\'adapter\'])\n    export_trained_model(detector, adapter, directory, identity, config)\n    del detector, adapter\n\n\ndef run_training(rows, config):\n    checks = preflight(config, rows)\n    if not checks.ready.all():\n        raise RuntimeError(\'Preflight failed:\\n\'+checks.loc[~checks.ready, [\'check\',\'detail\']].to_string(index=False))\n    output = Path(config[\'output\'])\n    output.mkdir(parents=True, exist_ok=True)\n    with FileLock(str(output/\'run.lock\'), timeout=0):\n        signature = {\'engine\': ENGINE_VERSION, \'config\': config, \'inputs\': fingerprint(rows),\n                     \'density_sha256\': digest(config[\'density_checkpoint\']) if config.get(\'density_checkpoint\') else None,\n                     \'source_sha256\': {p.name: digest(p) for p in Path(__file__).parent.glob(\'fog_*.py\')},\n                     \'versions\': {p: importlib.metadata.version(p) for p in (\'torch\',\'ultralytics\',\'transformers\',\'pycocotools\')}}\n        settings = output/\'settings.json\'\n        if settings.exists():\n            previous = json.loads(settings.read_text())\n            if previous != signature:\n                immutable = (\'config\', \'inputs\', \'density_sha256\', \'versions\')\n                compatible = all(previous.get(key) == signature.get(key) for key in immutable)\n                if compatible and previous.get(\'engine\') in (ENGINE_VERSION-1, ENGINE_VERSION):\n                    print(\'Migrating compatible checkpoint metadata to the current resumable engine.\', flush=True)\n                else:\n                    if previous.get(\'inputs\') != signature.get(\'inputs\'):\n                        raise ValueError(\'The input manifest changed since this run was started. The saved checkpoint belongs to a different dataset version; use a new output folder for the complete-data run. The old checkpoint is preserved.\')\n                    raise ValueError(\'Run configuration, inputs or package versions changed. Choose a new output folder to avoid mixing experiments.\')\n        atomic_json(settings, signature)\n        atomic_json(output/\'input-manifest.json\', rows)\n        parts = partitions(rows)\n        statuses = []\n        models = list(config[\'checkpoints\'])\n        for seed in config[\'seeds\']:\n            density = None\n            if \'predicted_density\' in config[\'arms\']:\n                density = fit_density(parts, config, output/f\'density-{seed}\', seed)\n                density_test_path = output/f\'density-{seed}\'/\'test-predictions.csv\'\n                if not density_test_path.exists():\n                    density_results = []\n                    with torch.inference_mode():\n                        for row in tqdm(parts[\'density_test\'], desc=\'Held-out density quality\', mininterval=2):\n                            estimate = float(density(image_tensor(row, config[\'size\'], config[\'device\'])).item())\n                            density_results.append({\'id\':row[\'id\'],\'dataset\':row[\'dataset\'],\'target_beta\':row[\'beta\'],\'predicted_beta\':estimate})\n                    pd.DataFrame(density_results, columns=[\'id\',\'dataset\',\'target_beta\',\'predicted_beta\']).to_csv(density_test_path,index=False)\n            # Rotate evaluation order across seeds to reduce fixed-order timing bias.\n            offset = config[\'seeds\'].index(seed) % len(models)\n            for name in models[offset:]+models[:offset]:\n                for arm in config[\'arms\']:\n                    directory = output/f\'{name}-{arm}-{seed}\'\n                    directory.mkdir(exist_ok=True)\n                    identity = {\'model\': name, \'arm\': arm, \'seed\': seed}\n                    status_file = directory/\'status.json\'\n                    if status_file.exists() and json.loads(status_file.read_text()).get(\'status\') == \'complete\':\n                        export_saved_checkpoint(name, directory, identity, config)\n                        statuses.append(json.loads(status_file.read_text())); continue\n                    detector = adapter = None\n                    try:\n                        seed_all(seed)\n                        detector = Detector(name, config[\'checkpoints\'][name], config[\'device\'], trainable=True)\n                        seed_all(seed)\n                        adapter = FogAdapter().to(config[\'device\'])\n                        trainable = sum(p.numel() for p in detector.model.parameters() if p.requires_grad)\n                        atomic_json(status_file, {**identity, \'status\': \'training\', \'trainable_detector_parameters\': trainable})\n                        fit_detector(detector, adapter, density, parts, arm, config, directory, seed)\n                        export_trained_model(detector, adapter, directory, identity, config)\n                        atomic_json(status_file, {**identity, \'status\': \'evaluating\'})\n                        summaries = []\n                        for dataset in sorted({r[\'dataset\'] for r in parts[\'test\']}):\n                            test_rows = sorted([r for r in parts[\'test\'] if r[\'dataset\'] == dataset], key=lambda r: r[\'id\'])\n                            if config[\'device\'].startswith(\'cuda\'):\n                                torch.cuda.reset_peak_memory_stats()\n                            predictions = evaluate_resumable(detector, adapter, density, arm, test_rows, config,\n                                                             directory/f\'{dataset}-predictions.jsonl\')\n                            peak = torch.cuda.max_memory_allocated()/2**20 if config[\'device\'].startswith(\'cuda\') else None\n                            for metric in summarize(test_rows, predictions):\n                                summaries.append({**identity, \'dataset\': dataset, **metric,\n                                    \'peak_gpu_mb\': peak, \'trainable_detector_parameters\': trainable,\n                                    \'trained_checkpoint_mb\': (directory/\'best.pt\').stat().st_size/2**20})\n                        pd.DataFrame(summaries).to_csv(directory/\'results.csv\', index=False)\n                        atomic_json(status_file, {**identity, \'status\': \'complete\'})\n                    except BaseException as error:\n                        atomic_json(status_file, {**identity, \'status\': \'interrupted\' if isinstance(error, KeyboardInterrupt) else \'error\',\n                                                 \'reason\': f\'{type(error).__name__}: {error}\'})\n                        # Fail visibly; rerunning with the same config resumes saved work.\n                        raise\n                    finally:\n                        del detector, adapter\n                        gc.collect()\n                        if torch.cuda.is_available():\n                            torch.cuda.empty_cache()\n                    statuses.append(json.loads(status_file.read_text()))\n        from fog_analytics import export_analytics\n        return export_analytics(output, rows), pd.DataFrame(statuses)\n', 'requirements-notebook.txt': 'jupyterlab\nnbformat\nnbclient\nmatplotlib\nnumpy\npandas\npillow\ntorch\ntorchvision\nultralytics\ntransformers\nijson\ngdown\n', 'requirements-fog-research.txt': '# Use the existing project environment; select a GPU-compatible torch build first.\n-r requirements-notebook.txt\npycocotools\nnuscenes-devkit\nshapely\npyquaternion\nscipy\n# DETR uses the official no_timm revision.\nfilelock\ntqdm\nkagglehub\n', 'requirements-colab.txt': "# Keep Colab's installed CUDA-compatible torch/torchvision pair.\n# Pin model/data APIs to versions present in the local research environment.\n-r requirements-fog-research.txt\nultralytics==8.4.155\ntransformers==5.17.0\npycocotools==2.0.11\nnuscenes-devkit==1.2.0\nkagglehub==1.0.2\n"}
for relative, source in EMBEDDED_FILES.items():
    target = ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
print('Restored', len(EMBEDDED_FILES), 'project files.')


## Install dependencies and verify GPU
Uses Colab’s existing PyTorch/CUDA installation. If pip requests a runtime restart, restart and rerun from the first cell.

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', '-r', str(ROOT/'requirements-colab.txt')])
import torch, torchvision
from torchvision.ops import nms
assert torch.cuda.is_available(), 'Select a GPU runtime before running the full study.'
nms(torch.tensor([[0., 0., 2., 2.]], device='cuda'), torch.tensor([1.], device='cuda'), .5)
print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=(ROOT/'colab-packages.txt').open('w'), check=True)


## Load your portable dataset ZIP
Extraction checks paths and free disk space. A completion marker avoids repeat extraction in the same runtime. Keep the ZIP in Drive for a fresh runtime.

In [ ]:
import urllib.request, zipfile, stat

if DATA_MODE == 'directory':
    for name in ('nuscenes_all.json', 'cityscapes.json'):
        if not (DATA_ROOT / 'data/fog' / name).is_file():
            raise FileNotFoundError(f'Missing {DATA_ROOT / "data/fog" / name}. Point DRIVE_DATA_DIR to the folder containing data/.')
    if shutil.disk_usage(ROOT).free < MIN_FREE_GIB * 2**30:
        raise OSError('Runtime free space is below the configured working-space reserve.')
    print('Using existing Drive data; no local extraction.')
else:
    
    if LOW_DISK_MODE:
        if DATA_ZIP_URL or not USE_DRIVE:
            raise ValueError('Low-disk mode requires the ZIP in mounted Drive, not a runtime-local download.')
        if not Path(DATA_ZIP_PATH).resolve().is_relative_to(Path('/content/drive').resolve()):
            raise ValueError('Set DATA_ZIP_PATH to the complete ZIP under /content/drive/.')
    if DATA_ZIP_URL:
        if not DATA_ZIP_URL.startswith('https://'):
            raise ValueError('Use a direct HTTPS URL.')
        bundle = ROOT/'downloaded-data.zip'
        if not bundle.exists():
            partial = bundle.with_suffix('.part')
            urllib.request.urlretrieve(DATA_ZIP_URL, partial)
            partial.replace(bundle)
    else:
        bundle = Path(DATA_ZIP_PATH)
    marker = ROOT/'.data-extracted'
    if not marker.exists():
        if not bundle.is_file():
            raise FileNotFoundError('Upload fog-colab-data.zip to the configured Drive path or set DATA_ZIP_URL. See the first cell.')
        with zipfile.ZipFile(bundle) as archive:
            members = archive.infolist()
            if LOW_DISK_MODE:
                needed = {'data/fog/nuscenes_all.json', 'data/fog/cityscapes.json'}
                if not needed.issubset(archive.namelist()):
                    raise ValueError('Low-disk mode needs both prepared datasets. Use the --include-fog bundle; no extra dataset download will be started.')
            for member in members:
                destination = (ROOT/member.filename).resolve()
                if not destination.is_relative_to(ROOT.resolve()) or not member.filename.startswith(('data/', '.cache/')) or stat.S_ISLNK(member.external_attr >> 16):
                    raise ValueError('Unsafe or unexpected ZIP member: ' + member.filename)
            required = sum(m.file_size for m in members)
            reserve = int(MIN_FREE_GIB * 2**30)
            free = shutil.disk_usage(ROOT).free
            print(f'Extraction: {required / 2**30:.2f} GiB; free: {free / 2**30:.2f} GiB; reserve: {MIN_FREE_GIB} GiB')
            if required + reserve > free:
                raise OSError(f'Need {(required + reserve) / 2**30:.2f} GiB FREE before extraction, including working space. Total disk capacity is not free space.')
            archive.extractall(ROOT)
        marker.write_text('complete')
    assert (ROOT/'data/fog/nuscenes_all.json').is_file(), 'ZIP must be produced by package_colab_data.py.'
    if LOW_DISK_MODE:
        assert (ROOT/'data/fog/cityscapes.json').is_file(), 'Complete bundle with Foggy Cityscapes required.'
    if shutil.disk_usage(ROOT).free < MIN_FREE_GIB * 2**30:
        raise OSError('Runtime free space is below the configured working-space reserve.')
    print('Portable nuScenes data is ready.')


## Load research modules and display resolved configuration

In [ ]:
from pathlib import Path
import json
import os
import sys
import importlib
import pandas as pd
from IPython.display import display, Image


def configure_environment():
    root = Path.cwd().resolve()
    if root.name == 'notebooks':
        root = root.parent
    support = str(root / 'notebooks/_support')
    if support not in sys.path:
        sys.path.insert(0, support)
    os.environ.setdefault('YOLO_CONFIG_DIR', str(root / '.ultralytics'))
    os.environ.setdefault('HF_HOME', str(root / '.cache/huggingface'))
    os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
    return root


ROOT = configure_environment()
from fog_data import cityscapes_manifest, resolve_foggy_images
from fog_analytics import export_analytics
from fog_prepare import prepare_nuscenes
from fog_research import read_manifest
import fog_training
fog_training = importlib.reload(fog_training)
from fog_training import default_config, preflight, run_training


# Resolve the configuration from the first cell; no hidden experiment overrides.
CONFIG = default_config(ROOT)
CONFIG.update(STUDY)
CONFIG['checkpoints'] = {name: value if name == 'DETR' else str(ROOT / value)
                         for name, value in MODEL_CHECKPOINTS.items()}
CONFIG['output'] = str(OUTPUT_ROOT / RUN_NAME)
CONFIG['manifest'] = str(DATA_ROOT / 'data/fog/nuscenes_all.json')
CONFIG['city_manifest'] = str(DATA_ROOT / 'data/fog/cityscapes.json')
CITY_ROOT = DATA_ROOT / 'data/fog/cityscapes'
DOWNLOAD_FOG_IMAGES = not LOW_DISK_MODE and DATA_MODE == 'zip'
NUSCENES_ROOT = DATA_ROOT / 'data/depth/datasets/nuscenes'
assert ACTION in {'train', 'preflight', 'analyze'}
display(pd.DataFrame([{'setting': key, 'value': str(value)}
                      for key, value in CONFIG.items()]))

## 2. Prepare data

Build or reuse the complete nuScenes and Foggy Cityscapes manifests.

In [ ]:
def prepare_manifests(root, config):
    if ACTION == 'analyze':
        return
    if not Path(config['manifest']).exists():
        prepare_nuscenes(NUSCENES_ROOT, config['manifest'])
    if not Path(config['city_manifest']).exists():
        source = resolve_foggy_images(root, CITY_ROOT, download=DOWNLOAD_FOG_IMAGES)
        if source is not None:
            cityscapes_manifest(source, config['city_manifest'])
        else:
            print('Foggy Cityscapes not found; density training is blocked.')


def load_rows(config):
    rows = []
    for key in ('manifest', 'city_manifest'):
        path = Path(config[key])
        if path.exists():
            print(f'Validating {path.name}')
            portable_rows = json.loads(path.read_text(encoding='utf-8'))
            for row in portable_rows:
                image_path = (path.parent / row['image']).resolve()
                os.utime(image_path, ns=(946684800000000000, 946684800000000000))
            rows.extend(read_manifest(path))
    return rows


def show_data_summary(rows, config):
    if not rows:
        return
    frame = pd.DataFrame(rows)
    display(frame.groupby(['dataset', 'split']).agg(
        images=('id', 'size'), scenes=('group', 'nunique'), known_beta=('beta', 'count')))
    coverage_path = Path(config['manifest']).with_suffix('.coverage.json')
    if coverage_path.exists():
        coverage = json.loads(coverage_path.read_text())
        display(pd.DataFrame([{
            'expected': coverage['expected_images'],
            'available': coverage['available_images'],
            'missing': len(coverage['missing_images']),
            'complete': coverage['complete']}]))


prepare_manifests(ROOT, CONFIG)
rows = load_rows(CONFIG)
show_data_summary(rows, CONFIG)

## 3. Validate

Preflight stops before training when data, models or settings are invalid.

In [ ]:
def validate_run(config, rows):
    checks = preflight(config, rows)
    display(checks)
    if not checks.ready.all():
        display(checks.loc[~checks.ready, ['check', 'detail']])
    return checks


checks = None if ACTION == 'analyze' else validate_run(CONFIG, rows)

## 4. Train and evaluate

Interrupt the kernel to checkpoint; rerun with the same output path to resume.

In [ ]:
def execute_analysis(config):
    return export_analytics(config['output'])


def execute_training(action, config, rows, checks):
    if action == 'preflight':
        return pd.DataFrame(), None
    if not checks.ready.all():
        print('Training is blocked by the checks above.')
        return pd.DataFrame(), None
    return run_training(rows, config)


def execute_run(action, config, rows, checks):
    if action == 'analyze':
        return execute_analysis(config), None
    return execute_training(action, config, rows, checks)


results, status = execute_run(ACTION, CONFIG, rows, checks)
if status is not None:
    display(status)

## 5. Review

Reports include accuracy, latency, density error, paired deltas and saved checkpoints.

In [ ]:
def show_metrics(results, output):
    if results.empty:
        print('No completed runs yet.')
        return
    display(results[results.density == 'all'].round(4))
    paired = pd.read_csv(Path(output) / 'paired-deltas.csv', dtype={'density': str})
    display(paired.loc[paired.density == 'all',
                       ['model', 'dataset', 'seed', 'arm', 'delta_mAP', 'delta_latency_ms']].round(4))
    density = pd.read_csv(Path(output) / 'density-quality.csv')
    if not density.empty:
        display(density.round(6))


def show_figures(output, results):
    if results.empty:
        return
    for pattern in ('*-comparison.png', '*-deltas.png', '*-classes.png', '*-density.png', '*-loss.png'):
        for path in sorted((Path(output) / 'figures').glob(pattern)):
            display(Image(filename=str(path), width=1000))


show_metrics(results, CONFIG['output'])
show_figures(CONFIG['output'], results)

## Notes

nuScenes boxes are projected 2D proxies. Foggy Cityscapes beta comes from filenames. Results are saved under the configured output path.

## Export results
Run after training, or after interrupting it. The ZIP includes checkpoints and may be large. With Drive enabled, it stays in Drive; otherwise the browser download starts.

In [ ]:
export_path = OUTPUT_ROOT / (RUN_NAME + '-results')
Path(CONFIG['output']).mkdir(parents=True, exist_ok=True)
shutil.copy2(ROOT/'colab-packages.txt', Path(CONFIG['output'])/'colab-packages.txt')
archive_path = shutil.make_archive(str(export_path), 'zip', CONFIG['output'])
print('Saved:', archive_path)
if not USE_DRIVE:
    from google.colab import files
    files.download(archive_path)
